In [1]:
import os
import json
import re
import string
import pandas as pd
import nltk

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\810350135\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
#1. import json dari folder
folder_path = r"C:\Users\810350135\Documents\DJPForum\28 April 2026"

all_data = []

for file in os.listdir(folder_path):
    if file.endswith(".json"):
        file_path = os.path.join(folder_path, file)

        with open(file_path, "r", encoding="utf-8-sig") as f:
            raw = json.load(f)

            if "data" in raw:
                all_data.extend(raw["data"])

df = pd.DataFrame(all_data)

print("Jumlah data awal:", df.shape)
print(df.head())
print(df.columns)

Jumlah data awal: (19552, 19)
  postid              handle  \
0  71074                 a.a   
1  70027                 a.a   
2  74511        abed.siahaan   
3  69012        abed.siahaan   
4  56232  ach.ariek.prasetya   

                                      question_title  \
0  Billing Deposit sudah dibayar namun belum masu...   
1  Apakah Pengajuan Surat Keterangan Tidak Dipung...   
2                   KENAPA LINK MELATI SELALU ERROR?   
3                         Tiket Melati Belum Dibalas   
4  Kamus cara baca buku besar  WP di  Coretax per...   

                                            question        question_date  \
0  Mohon Bantuan\n\nPembayaran deposit tanggal 15...  2026-01-19 04:43:22   
1  Apakah Pengajuan Surat Keterangan Tidak Dipung...  2026-01-05 02:58:47   
2  <p>Mohon dibantu apakah ada link alamat melati...  2026-04-07 01:49:01   
3  <p></p><dl class="ng-star-inserted" style="box...  2025-12-15 10:24:37   
4  Mohon bantuan dan arahannya, saat ini kami seb...  2

In [3]:
#2. pilih kolom penting
df = df[['postid','question_title','question',
         'question_date','category',
         'parent1','parent2','source']].copy()

In [4]:
#3. handle null value
df['question_title'] = df['question_title'].fillna('')
df['question'] = df['question'].fillna('')
df['category'] = df['category'].fillna('Unknown')

In [5]:
#4. convert tanggal
df['question_date'] = pd.to_datetime(df['question_date'], errors='coerce')

In [6]:
#5. hapus duplikat
df = df.drop_duplicates(subset=['question_title','question'])

In [7]:
#6. gabungkan title dan question
df['raw_text'] = df['question_title'] + " " + df['question']

In [8]:
#7. tax keywords
tax_keywords = {
    'npwp','nik','spt','pph','ppn',
    'aktivasi','coretax','faktur',
    'billing','deposit',
    'pkp','pemindahbukuan',
    'pendaftaran','status','perubahan'}

In [9]:
#8. tax dictionary normalization
tax_dict = {
    'wajib pajak':'wajib_pajak',
    'spt tahunan':'spt_tahunan',
    'spt masa':'spt_masa',
    'aktivasi akun':'aktivasi_akun',
    'faktur pajak':'faktur_pajak',
    'kode billing':'kode_billing',
    'login gagal':'login_gagal',
    'perubahan status':'perubahan_status'
}

In [10]:
#9. stopwords
stop_words = set(stopwords.words('indonesian'))

custom_stopwords = {
    'mohon','tolong','bantu','bantuan',
    'terima','kasih','selamat',
    'siang','pagi','malam',
    'saya','kami'
}

stop_words.update(custom_stopwords)

In [11]:
#10. stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

In [12]:
#11. cleaning function
def clean_text(text):
    
    text = str(text).lower()
    
    # hapus html
    text = re.sub(r'<.*?>', ' ', text)
    
    # hapus url
    text = re.sub(r'http\S+|www\S+', ' ', text)
    
    # normalisasi phrase dictionary
    for k,v in tax_dict.items():
        text = text.replace(k, v)
    
    # hapus angka
    text = re.sub(r'\d+', ' ', text)
    
    # hapus punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # rapikan spasi
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    
    result = []
    
    for w in words:
        
        # jika keyword pajak -> simpan
        if w in tax_keywords:
            result.append(w)
        
        # jika bukan stopword -> stem
        elif w not in stop_words:
            result.append(stemmer.stem(w))
    
    return ' '.join(result)

In [ ]:
#12. apply cleaning
df['clean_text'] = df['raw_text'].apply(clean_text)

In [ ]:
#13. filter teks pendek
df = df[df['clean_text'].str.split().str.len() >= 3]

print("Jumlah data bersih:", df.shape)

In [ ]:
#14. TFIDF dan bigram
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),     # unigram + bigram
    min_df=3,
    max_df=0.90,
    sublinear_tf=True
)

X = vectorizer.fit_transform(df['clean_text'])

print("Shape TFIDF:", X.shape)

In [ ]:
#15. simpan
df.to_csv("djp_forum_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
#16. cari jumlah cluster terbaik
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scores = {}

for k in range(5,16):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels)
    scores[k] = score

print(scores)

In [ ]:
#17. train final kmeans
k = 8

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df['cluster'] = kmeans.fit_predict(X)

In [ ]:
#18. auto topic label
import numpy as np

terms = vectorizer.get_feature_names_out()

def top_keywords_cluster(cluster_num, n=10):
    
    center = kmeans.cluster_centers_[cluster_num]
    top_idx = center.argsort()[-n:][::-1]
    
    return [terms[i] for i in top_idx]

for i in range(k):
    print("Cluster", i)
    print(top_keywords_cluster(i))
    print()

In [ ]:
#19. auto nama cluster
def auto_label(words):
    
    text = " ".join(words)

    if 'pendaftaran' in text or 'NPWP' in text:
        return 'pendaftaran NPWP'
    
    elif 'perubahan' in text or 'status' in text:
        return 'perubahan status'
    
    elif 'billing' in text or 'deposit' in text:
        return 'Billing Deposit'
    
    elif 'spt_tahunan' in text:
        return 'SPT Tahunan'
    
    elif 'aktivasi' in text or 'akun' in text:
        return 'aktivasi akun'
    
    elif 'faktur' in text:
        return 'Faktur Pajak'
    
    elif 'spt_masa' in text:
        return 'SPT Masa'
    
    elif 'pemindahbukuan' in text:
        return 'Pemindahbukuan'
    
    else:
        return 'Lainnya'

cluster_labels = {}

for i in range(k):
    words = top_keywords_cluster(i)
    cluster_labels[i] = auto_label(words)

print(cluster_labels)